# **Imports & logging**


In [0]:
import logging
from pyspark.sql.functions import avg, count, when, col

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# **Compute customer features & global averages (production version)**


In [0]:
# ============================================
# PRODUCTION: Gold Customer Features Table
# ============================================

df_orders = spark.table("silver_orders_clean")

# 1. Validate input table is not empty
if df_orders.count() == 0:
    raise ValueError("Input table 'silver_orders_clean' is empty. Job aborted.")

logging.info(f"Processing {df_orders.count()} orders")

# 2. Compute customer-level aggregates
customer_features = df_orders.groupBy("customer_id").agg(
    avg(when(col("delivery_delay_flag") == 1, 1).otherwise(0)).alias("customer_delay_rate"),
    avg("service_rating").alias("customer_avg_rating"),
    count("order_id").alias("customer_order_count"),
    avg(when(col("refund_flag") == 1, 1).otherwise(0)).alias("customer_refund_rate")
)

# 3. Write as Delta table (overwrite entire table – safe for daily full refresh)
customer_features.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_customer_features")

logging.info(f"Saved {customer_features.count()} customer feature rows to 'gold_customer_features'")

# 4. Compute global averages from the same source
global_delay_rate = df_orders.agg(avg("delivery_delay_flag")).collect()[0][0]
global_avg_rating = df_orders.agg(avg("service_rating")).collect()[0][0]

globals_dict = {
    "customer_delay_rate": global_delay_rate,
    "customer_avg_rating": global_avg_rating,
    "customer_order_count": 0,
    "customer_refund_rate": 0.0
}

# 5. Save global averages (overwrite)
spark.createDataFrame([globals_dict]) \
    .write \
    .mode("overwrite") \
    .saveAsTable("global_averages")

logging.info("Global averages updated in 'global_averages'")

# 6. Optional: show for verification (in job logs, not needed for display)
# But you can keep display for debugging when run interactively
if 'display' in globals():
    display(spark.table("global_averages"))

2026-04-27 10:51:12,326 - INFO - Processing 100000 orders
2026-04-27 10:51:16,079 - INFO - Saved 9000 customer feature rows to 'gold_customer_features'
2026-04-27 10:51:19,736 - INFO - Global averages updated in 'global_averages'


customer_avg_rating,customer_delay_rate,customer_order_count,customer_refund_rate
3.96589,0.23798,0,0.0
